# GOaT selection notebook

Run top to bottom on Colab. Every stage writes its artifacts to Google Drive
and skips itself when those artifacts already exist.

## Step 1. Open the Colab notebook

## Step 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Step 3. Git clone the repo, shallow

In [ ]:
![ -d /content/GOaT/.git ] || git clone --depth 1 https://github.com/champyod/GOaT.git /content/GOaT
!ls -lh /content/GOaT/model/src/goat_model 2>&1 | head -n 20
!ls -lh /content/GOaT/model/notebooks 2>&1 | head -n 20


## Step 4. Install dependencies

In [ ]:
!apt-get install -y -q libraqm0 > /dev/null 2>&1
# %pip install -q transformers datasets sentencepiece peft accelerate jiwer sacrebleu opencv-python-headless scipy psutil tqdm matplotlib
# %pip install -q paddleocr paddlepaddle
# %pip install -q git+https://github.com/clovaai/synthtiger.git
# Alternative with uv (uses lock):
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv sync --project /content/GOaT/model --extra ocr --extra mt --extra train


## Step 5. Import the package from the clone

In [ ]:
import sys
sys.path.insert(0, "/content/GOaT/model/src")
from goat_model import constants as c
print("goat_model at", c.__file__)


## Args (Drive — passed to every step)

In [ ]:
DRIVE = "/content/drive/MyDrive/GOaT"
MT_DATA = f"{DRIVE}/datasets/mt"
MT_TEST_DIR = f"{DRIVE}/datasets/mt/test"
OCR_EVAL_DIR = f"{DRIVE}/datasets/ocr"
RESULTS = f"{DRIVE}/results"
DATA_ROOT = f"{DRIVE}/data"
ART_MT = "/content/artifacts/mt_lora"
ART_OCR = "/content/artifacts/ocr"
SEED = 42
REPEATS = 5


## Step 6. Data — download

In [ ]:
%run /content/GOaT/model/notebooks/data/download_data.py --dataset scb-mt --out-dir /content/drive/MyDrive/GOaT/datasets/mt
%run /content/GOaT/model/notebooks/data/download_data.py --dataset flores200 --out-dir /content/drive/MyDrive/GOaT/datasets/mt/test
%run /content/GOaT/model/notebooks/data/download_data.py --dataset thaiocrbench --out-dir /content/drive/MyDrive/GOaT/datasets/ocr/thaiocrbench
%run /content/GOaT/model/notebooks/data/download_data.py --dataset thai-ocr-evaluation --out-dir /content/drive/MyDrive/GOaT/datasets/ocr/thai-ocr-evaluation


## Step 7. Selection MT

In [ ]:
%run /content/GOaT/model/notebooks/selection/select_mt.py --mt-test-dir /content/drive/MyDrive/GOaT/datasets/mt/test --output /content/drive/MyDrive/GOaT/results/mt_selection.json --repeats 5 --seed 42


## Step 8. Selection OCR

In [ ]:
%run /content/GOaT/model/notebooks/selection/select_ocr.py --ocr-eval-dir /content/drive/MyDrive/GOaT/datasets/ocr --output /content/drive/MyDrive/GOaT/results/ocr_selection.json --repeats 5 --seed 42


## Step 9. Train MT

In [ ]:
%run /content/GOaT/model/notebooks/training/train_mt.py --mt-dir /content/drive/MyDrive/GOaT/datasets/mt --selection /content/drive/MyDrive/GOaT/results/mt_selection.json --output /content/drive/MyDrive/GOaT/results/mt_training.json --out-root /content/artifacts/mt_lora --seed 42


## Step 10. Train OCR

In [ ]:
%run /content/GOaT/model/notebooks/training/train_ocr.py --selection /content/drive/MyDrive/GOaT/results/ocr_selection.json --data-root /content/drive/MyDrive/GOaT/data --output /content/drive/MyDrive/GOaT/results/ocr_training.json --out-root /content/artifacts/ocr --seed 42
